# hy-cuts L=4 — electric-cut O_FM(hz) at h_y = 0, 0.2, 0.4

Side-by-side Z-string order parameter along the electric cut (h_x=0.2, sweep h_z),
sign-free vs sign-full. Points are **end-of-training pooled** `O_FM_paratoric`
(`--final_eval_rounds 8`): h_y=0 from the Phase-B rerun, h_y=0.2/0.4 from the
2026-08-26 hy-cuts campaign (jobs 57623248-53 / 57623300-05) — identical 500-step
dual-basis protocol. S₂ and per-snapshot series follow from the replay jobs.

In [ ]:
# %% 1 CONFIG
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt

ROOT = Path("../../results")
HYS  = [0.0, 0.2, 0.4]                          # panel order
DIRS = {0.0: ROOT / "phaseB_rerun/up/L4",       # sign-free reference, same protocol
        0.2: ROOT / "hy_cuts_L4/up/hy0.2/L4",
        0.4: ROOT / "hy_cuts_L4/up/hy0.4/L4"}
ERR_X = 3                                       # bars x3 on order-parameter panels (house rule)
SAVE_FIGS, FIGS = False, Path("../figs")

In [ ]:
# %% 2 load: (hz, O_FM, err) per hy, from final-state JSONs
def ofm_curve(d, hy):
    rows = []
    for f in sorted(Path(d).glob("*.json")):
        j = json.loads(f.read_text())
        c, o = j["config"], j.get("observables", {})
        if abs(c.get("hy", 0.0) - hy) > 1e-9 or j.get("diverged") \
           or o.get("O_FM_paratoric") is None:
            continue
        rows.append((c["hz"], o["O_FM_paratoric"], o["O_FM_paratoric_err"]))
    hz, v, e = map(np.array, zip(*sorted(rows)))
    return hz, v, e

curves = {hy: ofm_curve(DIRS[hy], hy) for hy in HYS}
{hy: len(c[0]) for hy, c in curves.items()}

In [ ]:
# %% 3 figure: one panel per hy, sign-free curve as open reference
cpos = {0.0: 0.15, 0.2: 0.5, 0.4: 0.8}         # plasma keyed by hy
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharex=True, sharey=True)
for ax, hy in zip(axes, HYS):
    if hy:
        hz0, v0, e0 = curves[0.0]
        ax.errorbar(hz0, v0, ERR_X * e0, fmt="o", mfc="none", color="0.6",
                    ms=5, capsize=2, lw=1, label="$h_y=0$ ref")
    hz, v, e = curves[hy]
    ax.errorbar(hz, v, ERR_X * e, fmt="o", color=plt.cm.plasma(cpos[hy]),
                ms=5, capsize=2, lw=1, label=f"$h_y={hy}$")
    ax.set_title(f"$h_y = {hy}$")
    ax.set_xlabel("$h_z$")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
axes[0].set_ylabel(rf"$O_\mathrm{{FM}}$ (Z-string)  (bars $\times${ERR_X})")
fig.suptitle("3D TC, L=4 OBC, electric cut ($h_x=0.2$): transition vs $h_y$", y=1.03)
fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / "hy_cuts_L4_ofm_electric.png", dpi=300, bbox_inches="tight")
plt.show()

**Reading**: O_FM rises earlier in h_z as h_y grows — the y-field destabilizes the
topological phase, shifting the electric transition down (half-max: ≈0.29 at h_y=0/0.2,
≈0.27 at h_y=0.4; barely resolvable at 0.2 on this coarse grid). Grid refinement at
h_z=0.24/0.28 and the S₂/inflection locators from the snapshot replay sharpen h_c.

## S₂-Rényi locator (snapshot replay, step-500 states)

Central-plaquette S₂ from `eval_snapshots.py --topological --fm_sector electric`
(jobs 57644883-87). Deep topological anchor = exact 3·ln2; the S₂ collapse is the
independent transition locator. No h_y=0 replay series exists (pre-`--topological`
replays), so the anchor line stands in as the reference.

In [ ]:
# %% 4 S2 vs hz from the snapshot series (last snapshot per run)
S2LN2 = 3 * np.log(2)
def s2_curve(d, hy):
    rows = []
    for f in sorted(Path(d).glob("*.snapshots.json")):
        j = json.loads(f.read_text())
        c = j.get("config", {})
        if abs(c.get("hy", 0.0) - hy) > 1e-9:
            continue
        ser = [s for s in j.get("series", []) if "error" not in s and s.get("S2") is not None]
        if not ser:
            continue
        last = ser[-1]
        rows.append((c["hz"], last["S2"], last["S2_err"]))
    hz, v, e = map(np.array, zip(*sorted(rows)))
    return hz, v, e

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharex=True, sharey=True)
for ax, hy in zip(axes, [0.2, 0.4]):
    hz, v, e = s2_curve(DIRS[hy], hy)
    ax.axhline(S2LN2, color="0.75", lw=1, ls="--", label=r"$3\ln 2$ (topo)")
    ax.errorbar(hz, v, ERR_X * e, fmt="o", color=plt.cm.plasma(cpos[hy]),
                ms=5, capsize=2, lw=1, label=f"$h_y={hy}$")
    ax.set_title(f"$h_y = {hy}$")
    ax.set_xlabel("$h_z$")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
axes[0].set_ylabel(rf"$S_2$ (central plaquette)  (bars $\times${ERR_X})")
fig.suptitle("S$_2$ collapse along the electric cut", y=1.03)
fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / "hy_cuts_L4_s2_electric.png", dpi=300, bbox_inches="tight")
plt.show()

**Reading**: S₂ sits at the topological plateau (≈3ln2) through hz≈0.22, then collapses —
midpoint between 0.26–0.30 at h_y=0.2 vs 0.22–0.26 at h_y=0.4, consistent with the O_FM
shift of h_c down in h_z with growing h_y. §A drift check: last-100-step O_FM drift ≤0.04
at every point — all 12 runs converged, no 750-step extensions needed. Error-bar caveat:
S₂ at h_y≠0 has no phase-coherence diagnostic yet (audit MEDIUM) — treat bars as optimistic.